In [ ]:
import pandas as pd
import re
from collections import defaultdict

# =============================
# LOAD DATA
# =============================
file_path = "input.xlsx"   # <-- change if needed

master = pd.read_excel(file_path, sheet_name="Master Data")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master")

# =============================
# CLEAN NUMERIC COLUMNS
# =============================
num_cols = [
    "Sub Count", "Inventory_25", "Daily Plan",
    "Monthly Requirement 1", "Minimum Quantity", "Plan"
]

for c in num_cols:
    if c in master.columns:
        master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)

ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# =============================
# CYCLE TIME LOOKUP (CHILD PART → CYCLE TIME)
# =============================
cycle_time_map = dict(zip(ppm["Material"], ppm["Machine"]))

# =============================
# CONSTANTS
# =============================
ALLOWED_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}
MACHINE_CAPACITY = 3960  # 22 hrs × 3 days

# =============================
# HELPER: NORMALIZE MACHINE NAMES
# =============================
def normalize_machine(m):
    if not m or str(m).lower() == "nan":
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\-]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# STEP 1: FILTER DAILY PLAN > 0
# =============================
valid = master[master["Daily Plan"] > 0].copy()

# =============================
# STEP 2: AGGREGATE AT CHILD PART LEVEL
# =============================
agg = valid.groupby("Child Part").agg({
    "Daily Plan": "sum",                    # 🔥 TRUE REQUIRED QTY
    "Inventory_25": "first",
    "Minimum Quantity": "first",
    "Vertical Machines": lambda x: ",".join(x.astype(str))
}).reset_index()

agg.rename(columns={"Daily Plan": "Required Qty"}, inplace=True)

# =============================
# STEP 3: NET REQUIRED QTY
# =============================
agg["Net Required Qty"] = agg["Required Qty"] - agg["Inventory_25"]

# =============================
# STEP 4: CALCULATE TARGET LOAD (SMART BALANCING)
# =============================
mandatory_load = 0

for _, r in agg.iterrows():
    if r["Net Required Qty"] > 0:
        ct = cycle_time_map.get(r["Child Part"], 0)
        mandatory_load += r["Net Required Qty"] * ct

TARGET_LOAD = mandatory_load / len(ALLOWED_MACHINES) if mandatory_load > 0 else 0

# =============================
# TRACKING STRUCTURES
# =============================
machine_load = {m: 0 for m in ALLOWED_MACHINES}
machine_plan = defaultdict(list)
rejections = []

# =============================
# STEP 5: SMART ALLOCATION
# =============================
for _, row in agg.iterrows():

    child = row["Child Part"]
    net_qty = row["Net Required Qty"]

    cycle_time = cycle_time_map.get(child, 0)
    if cycle_time <= 0:
        continue

    # ---- PARSE MACHINES ----
    raw = str(row["Vertical Machines"])
    tokens = re.split(r"[,\|/\\\n]+", raw)
    vertical = [normalize_machine(t) for t in tokens if normalize_machine(t)]
    eligible = [m for m in vertical if m in ALLOWED_MACHINES]

    if not eligible:
        if net_qty > 0:
            rejections.append((child, "No eligible target machine"))
        continue

    # ---- DECIDE PRODUCTION QTY ----
    if net_qty > 0:
        qty_to_make = net_qty                # mandatory
    else:
        qty_to_make = abs(net_qty)           # balancing-only upper bound

    remaining_time = qty_to_make * cycle_time

    # ---- MACHINE SCORING FUNCTION ----
    def score(machine, alloc):
        return abs((machine_load[machine] + alloc) - TARGET_LOAD)

    eligible.sort(key=lambda m: score(
        m, min(MACHINE_CAPACITY - machine_load[m], remaining_time)
    ))

    for m in eligible:
        if remaining_time <= 0:
            break

        # Balancing-only rule
        if net_qty <= 0 and machine_load[m] >= TARGET_LOAD:
            continue

        available = MACHINE_CAPACITY - machine_load[m]
        if available <= 0:
            continue

        alloc_time = min(available, remaining_time)
        alloc_qty = alloc_time / cycle_time

        machine_load[m] += alloc_time
        remaining_time -= alloc_time

        machine_plan[m].append({
            "Child Part": child,
            "Quantity": round(alloc_qty, 2),
            "Time (min)": round(alloc_time, 2)
        })

    if net_qty > 0 and remaining_time > 0:
        rejections.append((child, "Capacity shortfall"))

# =============================
# DISPLAY RESULTS
# =============================
print("\n================ MACHINE-WISE PLAN ================\n")
for m in sorted(ALLOWED_MACHINES):
    print(f"🔧 {m}")
    if machine_plan[m]:
        display(pd.DataFrame(machine_plan[m]))
    else:
        print("No allocation")
    print("-" * 60)

print("\n================ MACHINE LOAD SUMMARY ================\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_load[m], 2),
        "Remaining (min)": round(MACHINE_CAPACITY - machine_load[m], 2)
    }
    for m in sorted(ALLOWED_MACHINES)
]))

print("\n================ REJECTIONS ================\n")
if rejections:
    display(pd.DataFrame(rejections, columns=["Child Part", "Reason"]))
else:
    print("✅ No rejections")


In [ ]:
import pandas as pd
import re
from collections import defaultdict

# =============================
# LOAD DATA
# =============================
file_path = "input.xlsx"   # change if needed

master = pd.read_excel(file_path, sheet_name="Master Data")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master")

# =============================
# CLEAN NUMERIC COLUMNS
# =============================
num_cols = [
    "Inventory_25", "Daily Plan",
    "Monthly Requirement 1", "Minimum Quantity", "Plan"
]
for c in num_cols:
    if c in master.columns:
        master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)

ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# =============================
# CYCLE TIME LOOKUP (SECONDS → MINUTES)
# =============================
cycle_time_sec = dict(zip(ppm["Material"], ppm["Machine"]))
cycle_time_min = {k: v / 60 for k, v in cycle_time_sec.items()}

# =============================
# CONSTANTS
# =============================
ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
MACHINE_CAPACITY = 3960  # minutes (22 hrs × 3 days)

# =============================
# HELPER
# =============================
def normalize_machine(m):
    if not m or str(m).lower() == "nan":
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\-]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# STEP 1: FILTER DAILY PLAN > 0
# =============================
valid = master[master["Daily Plan"] > 0].copy()

# =============================
# STEP 2: AGGREGATE REQUIRED QTY BY CHILD PART
# =============================
agg = valid.groupby("Child Part").agg({
    "Daily Plan": "sum",
    "Inventory_25": "first",
    "Vertical Machines": lambda x: ",".join(x.astype(str))
}).reset_index()

agg.rename(columns={"Daily Plan": "Required Qty"}, inplace=True)
agg["Net Required Qty"] = agg["Required Qty"] - agg["Inventory_25"]

# =============================
# TRACKING
# =============================
machine_load = {m: 0 for m in ALLOWED_MACHINES}
machine_plan = defaultdict(list)
rejections = []

# =============================
# STEP 3: SIMPLE ASSIGNMENT (NO BALANCING)
# =============================
for _, row in agg.iterrows():

    child = row["Child Part"]
    net_qty = row["Net Required Qty"]

    if net_qty <= 0:
        continue

    ct = cycle_time_min.get(child, 0)
    if ct <= 0:
        rejections.append((child, "Missing cycle time"))
        continue

    required_time = net_qty * ct

    # Parse machines
    raw = str(row["Vertical Machines"])
    tokens = re.split(r"[,\|/\\\n]+", raw)
    vertical = [normalize_machine(t) for t in tokens if normalize_machine(t)]
    eligible = [m for m in ALLOWED_MACHINES if m in vertical]

    if not eligible:
        rejections.append((child, "No eligible machine"))
        continue

    remaining_time = required_time

    for m in eligible:
        if remaining_time <= 0:
            break

        available = MACHINE_CAPACITY - machine_load[m]
        if available <= 0:
            continue

        alloc_time = min(available, remaining_time)
        alloc_qty = alloc_time / ct

        machine_load[m] += alloc_time
        remaining_time -= alloc_time

        machine_plan[m].append({
            "Child Part": child,
            "Quantity": int(alloc_qty),
            "Time Used (min)": round(alloc_time, 2)
        })

    if remaining_time > 0:
        rejections.append((child, "Capacity shortfall"))

# =============================
# DISPLAY RESULTS
# =============================
print("\n========== MACHINE-WISE PLAN ==========\n")
for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    if machine_plan[m]:
        display(pd.DataFrame(machine_plan[m]))
    else:
        print("No allocation")
    print("-" * 60)

print("\n========== MACHINE LOAD SUMMARY ==========\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_load[m], 2),
        "Remaining (min)": round(MACHINE_CAPACITY - machine_load[m], 2)
    }
    for m in ALLOWED_MACHINES
]))

print("\n========== REJECTIONS ==========\n")
if rejections:
    display(pd.DataFrame(rejections, columns=["Child Part", "Reason"]))
else:
    print("✅ No rejections")
